# Central Virginia LiDAR — VGIN Tile Query for Fluvanna County

Queries the **VGIN REST API** for all LAZ tile records that intersect a bounding box,  
then writes the results as a CSV directly to the project S3 bucket.

**Output:** `s3://central-virginia-tree-canopy-project/data/outputs/<COUNTY>/CentralVA_LiDAR_<COUNTY>.csv`

---
| Parameter | Value |
|:---|:---|
| VGIN endpoint | Virginia LiDAR Downloads MapServer/1 |
| Output format | VGIN attribute export CSV (14 columns) |
| AWS region | us-east-1 |
| IAM role | TreeCanopyPipelineRole (attached to this notebook instance) |

## Cell 1 — Configuration

Edit the values in this cell to change the study area or output location.  
All other cells run without modification.

In [1]:
# ── Study area bounding box (WGS84 degrees: lon_min, lat_min, lon_max, lat_max) ──
BBOX = (-78.4916, 37.6905, -78.0633, 38.0064)   # Fluvanna County

# ── Output S3 location ────────────────────────────────────────────────────────
S3_BUCKET = "central-virginia-tree-canopy-project"
S3_PREFIX = "data/inputs/Fluvanna/"       # must end with /
OUTPUT_FILENAME = "CentralVA_LiDAR_Fluvanna.csv"

# ── Optional: also save a local copy on the notebook instance ─────────────────
SAVE_LOCAL = False
LOCAL_PATH = f"/home/ec2-user/SageMaker/{OUTPUT_FILENAME}"

# ── AWS region ────────────────────────────────────────────────────────────────
AWS_REGION = "us-east-1"

print("Configuration loaded.")
print(f"  Bounding box : {BBOX}")
print(f"  S3 output    : s3://{S3_BUCKET}/{S3_PREFIX}{OUTPUT_FILENAME}")
if SAVE_LOCAL:
    print(f"  Local copy   : {LOCAL_PATH}")

Configuration loaded.
  Bounding box : (-78.4916, 37.6905, -78.0633, 38.0064)
  S3 output    : s3://central-virginia-tree-canopy-project/data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna.csv


## Cell 2 — Imports and Constants

In [2]:
import csv
import io
import json
import re
import time
import urllib.parse
import urllib.request

import boto3

import itertools

# ── VGIN REST endpoint ────────────────────────────────────────────────────────
VGIN_URL = (
    "https://vginmaps.vdem.virginia.gov/arcgis/rest/services/Download/"
    "Virginia_LiDAR_Downloads/MapServer/1/query"
)

PAGE_SIZE = 1000   # max features per VGIN API request

# CSV column names matching the VGIN attribute export format exactly
CSV_FIELDNAMES = [
    "VComment", "ProjectYea", "OBJECTID_1", "VLPID", "TileName",
    "PointCloud", "PointClo_1", "PointClo_2",
    "DEMHost", "DEMDownloa",
    "ShapeSTAre", "ShapeSTLen", "Shape_Length", "Shape_Area",
]

SHAPE_AREA_KEY   = "Shape.STArea()"
SHAPE_LENGTH_KEY = "Shape.STLength()"

print("Imports complete. All dependencies are available.")

Imports complete. All dependencies are available.


## Cell 3 — Helper Functions

In [3]:
def _reformat_anchor(html: str) -> str:
    """
    Convert a VGIN REST anchor (backslash-escaped quotes) to the
    double-double-quote form that csv.writer encodes correctly per RFC 4180.
    """
    if not html:
        return ""
    url_m   = re.search(r'href=\\"([^\\]+)\\"', html)
    label_m = re.search(r'>([^<]+)<', html)
    if url_m and label_m:
        return f'<a href=""{url_m.group(1)}"">{label_m.group(1)}</a>'
    url_m2 = re.search(r'href="([^"]+)"', html)
    if url_m2 and label_m:
        return f'<a href=""{url_m2.group(1)}"">{label_m.group(1)}</a>'
    return html


def _extract_url(html_or_url: str) -> str:
    """Return the bare href URL from an HTML anchor string."""
    if not html_or_url:
        return ""
    m = re.search(r'href=["\']([^"\']+)["\']', html_or_url)
    return m.group(1) if m else html_or_url


def feature_to_csv_row(feature: dict) -> dict:
    """Map a VGIN REST API feature to a CSV row dict matching CSV_FIELDNAMES."""
    attr         = feature.get("attributes", {})
    shape_area   = attr.get(SHAPE_AREA_KEY,   "")
    shape_length = attr.get(SHAPE_LENGTH_KEY, "")
    return {
        "VComment":     attr.get("VComment", ""),
        "ProjectYea":   attr.get("ProjectYear", ""),
        "OBJECTID_1":   attr.get("OBJECTID", ""),
        "VLPID":        attr.get("VLPID", ""),
        "TileName":     attr.get("TileName", ""),
        "PointCloud":   attr.get("PointCloudHost", ""),
        "PointClo_1":   attr.get("PointCloudFormat", ""),
        "PointClo_2":   _reformat_anchor(attr.get("PointCloudDownload", "")),
        "DEMHost":      attr.get("DEMHost", ""),
        "DEMDownloa":   _reformat_anchor(attr.get("DEMDownload", "")),
        "ShapeSTAre":   shape_area,
        "ShapeSTLen":   shape_length,
        "Shape_Length": shape_length,
        "Shape_Area":   shape_area,
    }


def rows_to_csv_string(rows: list) -> str:
    """Serialise row dicts to a CSV string using CSV_FIELDNAMES column order."""
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue()


print("Helper functions defined.")

Helper functions defined.


## Cell 4 — Query VGIN REST API

In [4]:
lon_min, lat_min, lon_max, lat_max = BBOX

print(f"Querying VGIN REST API …")
print(f"  Bounding box : lon [{lon_min}, {lon_max}]  lat [{lat_min}, {lat_max}]")
print(f"  Endpoint     : {VGIN_URL}")
print()

all_features = []
offset = 0

while True:
    params = urllib.parse.urlencode({
        "geometry":          f"{lon_min},{lat_min},{lon_max},{lat_max}",
        "geometryType":      "esriGeometryEnvelope",
        "inSR":              "4326",
        "spatialRel":        "esriSpatialRelIntersects",
        "outFields":         "*",
        "returnGeometry":    "false",
        "resultOffset":      offset,
        "resultRecordCount": PAGE_SIZE,
        "f":                 "json",
    })

    with urllib.request.urlopen(VGIN_URL + "?" + params, timeout=30) as r:
        data = json.load(r)

    features = data.get("features", [])
    all_features.extend(features)
    print(f"  Page offset {offset:>4} : {len(features)} feature(s) returned")

    if len(features) < PAGE_SIZE:
        break

    offset += PAGE_SIZE
    time.sleep(0.2)

print()
print(f"Total tiles found : {len(all_features)}")

Querying VGIN REST API …
  Bounding box : lon [-78.4916, -78.0633]  lat [37.6905, 38.0064]
  Endpoint     : https://vginmaps.vdem.virginia.gov/arcgis/rest/services/Download/Virginia_LiDAR_Downloads/MapServer/1/query

  Page offset    0 : 783 feature(s) returned

Total tiles found : 783


## Cell 5 — Build CSV and Display Summary

In [5]:
import pandas as pd

if not all_features:
    raise RuntimeError("No tiles returned by VGIN for the given bounding box. "
                       "Check the BBOX coordinates and try again.")

rows     = [feature_to_csv_row(f) for f in all_features]
csv_text = rows_to_csv_string(rows)

# Display a summary table
df = pd.read_csv(io.StringIO(csv_text))

years = sorted(df["ProjectYea"].dropna().astype(int).unique())
print("=" * 60)
print(f"  Tiles found   : {len(df)}")
print(f"  Project years : {', '.join(str(y) for y in years)}")
print(f"  Columns       : {list(df.columns)}")
print("=" * 60)
print()

# Show tile names and LPC download URLs
display_cols = ["TileName", "ProjectYea", "VLPID"]
df[display_cols].sort_values("TileName")

  Tiles found   : 783
  Project years : 2014, 2016, 2019
  Columns       : ['VComment', 'ProjectYea', 'OBJECTID_1', 'VLPID', 'TileName', 'PointCloud', 'PointClo_1', 'PointClo_2', 'DEMHost', 'DEMDownloa', 'ShapeSTAre', 'ShapeSTLen', 'Shape_Length', 'Shape_Area']



,TileName,ProjectYea,VLPID
656,17SPB500940,2019,39
650,17SPB500955,2019,39
644,17SPB500970,2019,39
657,17SPB515940,2019,39
651,17SPB515955,2019,39
...,...,...,...
608,S13_6802_20,2016,29
620,S13_6802_30,2016,29
619,S13_6802_40,2016,29
609,S13_6803_10,2016,29


## Cell 6 — Write CSV to S3 with BLOCK CHUNKING

In [6]:
s3_key   = S3_PREFIX + OUTPUT_FILENAME
s3_body  = csv_text.encode("utf-8")

# Setup variables for chunking
chunk_size = 500
base_filename, file_extension = OUTPUT_FILENAME.rsplit(".", 1)

s3_client = boto3.client("s3", region_name=AWS_REGION)

# Create alphabetical suffix generator (aa, ab, ac... zz)
alphabet = "abcdefghijklmnopqrstuvwxyz"
suffix_generator = (
    f"{a}{b}" for a, b in itertools.product(alphabet, alphabet)
)

print("\nStarting chunked S3 upload...")
print("-" * 60)

# Loop through the DataFrame in blocks of 500 rows
for i in range(0, len(df), chunk_size):
    # Slice the exact row range from your DataFrame
    df_chunk = df.iloc[i : i + chunk_size]

    # Convert the current 500-row chunk back into CSV text in memory
    csv_buffer = io.StringIO()
    df_chunk.to_csv(csv_buffer, index=False)
    s3_body_chunk = csv_buffer.getvalue().encode("utf-8")

    # Grab the next suffix pair (e.g., 'aa', then 'ab')
    part_suffix = next(suffix_generator)

    # Build the customized chunk name: CentralVA_LiDAR_Albemarle-part_aa.csv
    chunk_filename = f"{base_filename}-part_{part_suffix}.{file_extension}"
    s3_key = S3_PREFIX + chunk_filename

    # Upload this specific chunk object to S3
    s3_client.put_object(
        Bucket=S3_BUCKET,
        Key=s3_key,
        Body=s3_body_chunk,
        ContentType="text/csv",
    )

    print(f"Uploaded: s3://{S3_BUCKET}/{s3_key}")
    print(
        f"  Size : {len(s3_body_chunk):,} bytes  |  Rows : {len(df_chunk)}"
    )

print("-" * 60)
print("All pieces successfully chunked and uploaded to S3.")


Starting chunked S3 upload...
------------------------------------------------------------
Uploaded: s3://central-virginia-tree-canopy-project/data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna-part_aa.csv
  Size : 266,856 bytes  |  Rows : 500
Uploaded: s3://central-virginia-tree-canopy-project/data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna-part_ab.csv
  Size : 144,694 bytes  |  Rows : 283
------------------------------------------------------------
All pieces successfully chunked and uploaded to S3.


## Cell 7 — (Optional) Save Local Copy

In [7]:
if SAVE_LOCAL:
    with open(LOCAL_PATH, "w", newline="", encoding="utf-8") as f:
        f.write(csv_text)
    print(f"Local copy saved : {LOCAL_PATH}")
else:
    print("SAVE_LOCAL is False — skipping local file write.")

SAVE_LOCAL is False — skipping local file write.


## Cell 8 — Verify S3 Object Exists

In [8]:
response = s3_client.list_objects_v2(
    Bucket = S3_BUCKET,
    Prefix = S3_PREFIX,
)

print(f"Objects in s3://{S3_BUCKET}/{S3_PREFIX}")
print("-" * 70)
for obj in response.get("Contents", []):
    size_kb = obj["Size"] / 1024
    modified = obj["LastModified"].strftime("%Y-%m-%d %H:%M UTC")
    print(f"  {obj['Key']:<55}  {size_kb:6.1f} KB  {modified}")

Objects in s3://central-virginia-tree-canopy-project/data/inputs/Fluvanna/
----------------------------------------------------------------------
  data/inputs/Fluvanna/                                       0.0 KB  2026-07-16 13:36 UTC
  data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna-part_aa.csv   260.6 KB  2026-07-17 16:49 UTC
  data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna-part_ab.csv   141.3 KB  2026-07-17 16:49 UTC
  data/inputs/Fluvanna/CentralVA_LiDAR_Fluvanna.csv         402.6 KB  2026-07-16 13:36 UTC
